[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S21_feature_engineering.ipynb)

# Sesión 21 · Feature engineering

**Módulo 5: Machine Learning** · ⏱️ Duración estimada: 60 a 90 minutos (el módulo 5 prevé 1 o 2 días por sesión)

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Convertir categorías en números con one-hot, sin romperte ante categorías nuevas.
2. Escalar variables numéricas ajustando el escalador solo con el entrenamiento.
3. Crear variables nuevas (proporciones, logaritmos, tramos) y medir si ayudan.
4. Detectar una fuga de información (*data leakage*) antes de que te engañe.
5. Unir todo en un `Pipeline` con `ColumnTransformer` y validarlo sin fugas.

## 📋 Qué debes saber antes
Sesiones 17 a 20: partición en entrenamiento y prueba, regresión logística, AUC, validación cruzada y `GridSearchCV`.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica: solicitudes de préstamo ----------
_n = 3000
_emp = rng.choice(["dependiente", "independiente", "informal"], _n, p=[0.55, 0.3, 0.15])
_reg = rng.choice(["Lima", "Norte", "Sur", "Centro", "Oriente"], _n, p=[0.4, 0.2, 0.15, 0.15, 0.1])
_prod = rng.choice(["personal", "vehicular", "hipotecario"], _n, p=[0.5, 0.3, 0.2])
_ing = np.round(rng.lognormal(8.0, 0.7, _n), -1)
_deuda = np.round(_ing * rng.gamma(2, 2.0, _n), -1)
_ant = rng.integers(0, 241, _n)
_edad = rng.integers(21, 71, _n)
_logit = (-6.0 + 0.8 * (_deuda / _ing) - 0.9 * np.log(_ing / 3000) - 0.004 * _ant
          + pd.Series(_emp).map({"dependiente": 0, "independiente": 0.4, "informal": 1.0}).values
          + pd.Series(_reg).map({"Lima": 0, "Norte": 0.2, "Sur": 0, "Centro": 0.3, "Oriente": 0.7}).values
          + pd.Series(_prod).map({"personal": 0.5, "vehicular": 0, "hipotecario": -0.7}).values)
_y = (rng.random(_n) < 1 / (1 + np.exp(-_logit))).astype(int)
_mora = np.where((_y == 1) & (rng.random(_n) < 0.9), rng.integers(15, 121, _n),
                 np.where(rng.random(_n) < 0.08, rng.integers(1, 15, _n), 0))
prestamos = pd.DataFrame({
    "tipo_empleo": _emp, "region": _reg, "producto": _prod, "ingreso_mensual": _ing, "deuda_total": _deuda,
    "antiguedad_meses": _ant, "edad": _edad, "dias_mora_actual": _mora, "incumple": _y,
})
solicitudes_nuevas = pd.DataFrame({
    "tipo_empleo": ["dependiente", "informal", "jubilado", "independiente"],
    "region": ["Amazonas", "Lima", "Sur", "Norte"],
    "producto": ["personal", "vehicular", "hipotecario", "personal"],
    "ingreso_mensual": [2500.0, 1800.0, 3200.0, 6000.0],
    "deuda_total": [9000.0, 2000.0, 15000.0, 4000.0],
    "antiguedad_meses": [24, 6, 0, 120],
    "edad": [29, 41, 66, 50],
})
CATEGORICAS = ["tipo_empleo", "region", "producto"]
NUMERICAS = ["ingreso_mensual", "deuda_total", "antiguedad_meses", "edad"]
NUEVAS_NUM = ["log_ingreso", "ratio_deuda", "antiguedad_meses", "edad"]
TRAMOS = ["hasta 30", "31-45", "46-60", "más de 60"]

_D = copy.deepcopy({"prestamos": prestamos, "solicitudes_nuevas": solicitudes_nuevas})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _auc(real, prob):
    """AUC con la fórmula de rangos de Mann-Whitney (sin scikit-learn); los empates reciben el rango promedio."""
    real, prob = [int(v) for v in real], [float(v) for v in prob]
    orden = sorted(range(len(prob)), key=lambda i: prob[i])
    rangos = [0.0] * len(prob)
    i = 0
    while i < len(orden):
        j = i
        while j + 1 < len(orden) and prob[orden[j + 1]] == prob[orden[i]]:
            j += 1
        for k in range(i, j + 1):
            rangos[orden[k]] = (i + j) / 2 + 1
        i = j + 1
    pos = [rangos[i] for i in range(len(real)) if real[i] == 1]
    n_pos, n_neg = len(pos), len(real) - len(pos)
    return (math.fsum(pos) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def _particion():
    """El X_train, X_test, y_train, y_test del alumno, si existen y son coherentes."""
    v = [globals().get(n) for n in ("X_train", "X_test", "y_train", "y_test")]
    if all(isinstance(a, pd.DataFrame) for a in v[:2]) and all(isinstance(a, pd.Series) for a in v[2:]) and len(v[0]) == len(v[2]) and len(v[1]) == len(v[3]):
        return v
    return None


def _auc_modelo(m, X, y):
    return _auc(y.tolist(), m.predict_proba(X)[:, 1])


def _uno_caliente(df, categorias):
    """One-hot hecho a mano: una columna por (variable, categoría), 1.0 si coincide."""
    return np.array([[1.0 if fila[c] == v else 0.0 for c in CATEGORICAS for v in categorias[c]]
                     for _, fila in df.iterrows()]).reshape(len(df), -1)


def _categorias_train(xs):
    return {c: sorted(set(xs[c])) for c in CATEGORICAS}


def _escalar(A, B):
    """Escala B con la media y la desviación (ddof=0) de A, columna por columna."""
    A, B = np.asarray(A, dtype=float), np.asarray(B, dtype=float)
    medias = [statistics.fmean(A[:, j]) for j in range(A.shape[1])]
    desv = [statistics.pstdev(A[:, j]) for j in range(A.shape[1])]
    return np.column_stack([(B[:, j] - medias[j]) / desv[j] for j in range(B.shape[1])])


def _variables_ref(df):
    """Variables nuevas recalculadas fila por fila con math."""
    def tramo(e):
        return TRAMOS[0] if e <= 30 else TRAMOS[1] if e <= 45 else TRAMOS[2] if e <= 60 else TRAMOS[3]
    return ([d / i for d, i in zip(df["deuda_total"], df["ingreso_mensual"])],
            [math.log(1 + i) for i in df["ingreso_mensual"]],
            [tramo(e) for e in df["edad"]])


def _auc_logistica(xs, ys, xt, yt, columnas):
    from sklearn.linear_model import LogisticRegression
    a, b = xs[columnas].to_numpy(float), xt[columnas].to_numpy(float)
    m = LogisticRegression(max_iter=1000).fit(_escalar(a, a), ys)
    return _auc(yt.tolist(), m.predict_proba(_escalar(a, b))[:, 1])


def _cv_manual(modelo, X, y):
    """Validación cruzada escrita a mano: clona, entrena y evalúa en cada pliegue."""
    from sklearn.base import clone
    from sklearn.model_selection import StratifiedKFold
    notas = []
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(X, y):
        m = clone(modelo).fit(X.iloc[tr], y.iloc[tr])
        notas.append(_auc(y.iloc[va].tolist(), m.predict_proba(X.iloc[va])[:, 1]))
    return statistics.fmean(notas)


def _usa_fuga(pipe):
    prep = pipe.named_steps.get("prep") if hasattr(pipe, "named_steps") else None
    cols = [c for _, _, cs in getattr(prep, "transformers", []) for c in (cs if isinstance(cs, list) else [cs])]
    return any(_h(c) == "2a0dfcbf7860b36a1f67da25f93316daf70eb3501a68076538d070e992e08286" for c in cols) or "incumple" in cols


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    _sin_cambios_df(r, "prestamos", "solicitudes_nuevas")
    p = _particion()
    if p is None:
        r.mal("Faltan `X_train`, `X_test`, `y_train` o `y_test`.")
        r.fin()
        return
    xs, xt, ys, yt = p
    if len(xt) != 750 or "incumple" in xs.columns or len(xs.columns) != 8 or abs(ys.mean() - yt.mean()) > 0.005:
        r.mal("La partición debería dejar 25 % para prueba, estratificada, con todas las columnas menos `incumple`.")
    else:
        r.ok("La partición es correcta.")
    cats = _categorias_train(xs)
    nombres = [f"{c}_{v}" for c in CATEGORICAS for v in cats[c]]
    cod = r.var("codificador")
    if cod is not _FALTA:
        if type(cod).__name__ != "OneHotEncoder" or not hasattr(cod, "categories_"):
            r.mal("`codificador` debería ser un `OneHotEncoder` ya ajustado.")
        elif cod.handle_unknown != "ignore":
            r.mal("`codificador` debería usar `handle_unknown=\"ignore\"` para tolerar categorías nuevas.")
        elif [list(map(str, c)) for c in cod.categories_] != [cats[c] for c in CATEGORICAS]:
            r.mal("`codificador` no aprendió las categorías de `X_train[CATEGORICAS]`: ajústalo solo con el entrenamiento.")
        else:
            r.ok("`codificador` aprendió las categorías del entrenamiento.")
    nom = r.var("nombres_onehot")
    if nom is not _FALTA:
        if not isinstance(nom, list):
            r.mal(f"`nombres_onehot` es de tipo {type(nom).__name__} y se esperaba una lista (usa `list(...)`).")
        elif len(nom) != len(nombres):
            r.mal(f"`nombres_onehot` tiene {len(nom)} nombres y se esperaban {len(nombres)}.")
        elif [str(x) for x in nom] != nombres:
            r.mal("`nombres_onehot` no coincide con `codificador.get_feature_names_out()`.")
        else:
            r.ok("`nombres_onehot` es correcto.")
    ct = r.var("cat_train")
    if ct is not _FALTA:
        ref = _uno_caliente(xs, cats)
        if not isinstance(ct, pd.DataFrame):
            r.mal(f"`cat_train` es de tipo {type(ct).__name__} y se esperaba un DataFrame.")
        elif ct.shape != ref.shape:
            r.mal(f"`cat_train` tiene forma {ct.shape} y se esperaba {ref.shape}.")
        elif [str(c) for c in ct.columns] != nombres or not ct.index.equals(xs.index):
            r.mal("`cat_train` debería usar `nombres_onehot` como columnas y el índice de `X_train`.")
        elif not np.array_equal(ct.to_numpy(float), ref):
            r.mal("Los valores de `cat_train` no coinciden con la codificación de `X_train[CATEGORICAS]`.")
        else:
            r.ok("`cat_train` es correcto.")
    cn = r.var("cat_nuevas")
    if cn is not _FALTA:
        ref = _uno_caliente(solicitudes_nuevas, cats)
        cn = np.asarray(cn.toarray() if hasattr(cn, "toarray") else cn, dtype=float)
        if cn.shape != ref.shape:
            r.mal(f"`cat_nuevas` tiene forma {cn.shape} y se esperaba {ref.shape}.")
        elif not np.array_equal(cn, ref):
            r.mal("Los valores de `cat_nuevas` no coinciden: usa `codificador.transform` sobre `solicitudes_nuevas[CATEGORICAS]`.")
        else:
            r.ok("`cat_nuevas` es correcto: las categorías desconocidas quedan en ceros.")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_unos_fila0": "b4d4f68c2268549cada66d24ae3a1902d4152a98ad614bbf891edf235ef99218",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    p = _particion()
    if p is None:
        r.mal("Primero resuelve el ejercicio 1.")
        r.fin()
        return
    xs, xt, ys, yt = p
    e = r.var("escalador")
    if e is not _FALTA:
        if type(e).__name__ != "StandardScaler" or not hasattr(e, "mean_"):
            r.mal("`escalador` debería ser un `StandardScaler` ya ajustado.")
        elif len(e.mean_) != len(NUMERICAS) or not _cerca_lista(e.mean_, [statistics.fmean(xs[c]) for c in NUMERICAS], 1e-6):
            r.mal("`escalador` no tiene las medias de `X_train[NUMERICAS]`: ajústalo solo con el entrenamiento y esas columnas.")
        else:
            r.ok("`escalador` se ajustó con el entrenamiento.")
    a = xs[NUMERICAS].to_numpy(float)
    for nombre, datos in (("num_train", xs), ("num_test", xt)):
        v = r.var(nombre)
        if v is _FALTA:
            continue
        ref = _escalar(a, datos[NUMERICAS].to_numpy(float))
        v = np.asarray(v, dtype=float)
        if v.shape != ref.shape:
            r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {ref.shape}.")
        elif np.allclose(v, ref, atol=1e-6):
            r.ok(f"`{nombre}` es correcto.")
        elif nombre == "num_test" and np.allclose(v.mean(axis=0), 0, atol=1e-9):
            r.mal("`num_test` tiene media exactamente 0: parece escalado con sus propias medias. Usa el `escalador` del entrenamiento.")
        else:
            r.mal(f"Los valores de `{nombre}` no coinciden: transfórmalo con `escalador`.")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_media_test_cero": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    f = r.funcion("agregar_variables")
    if f is not _FALTA:
        prueba = pd.DataFrame({"ingreso_mensual": [2000.0, 5000.0, 1500.0, 3000.0], "deuda_total": [5000.0, 0.0, 6000.0, 300.0],
                               "edad": [30, 45, 61, 60], "region": ["Lima", "Norte", "Lima", "Centro"]}, index=[10, 20, 30, 40])
        casos = [("con edades en los límites de los tramos", prueba), ("con un DataFrame vacío", prueba.iloc[0:0])]
        for texto, df in casos:
            original = df.copy()
            try:
                res = f(df)
            except Exception as ex:
                r.mal(f"`agregar_variables` {texto} lanzó {type(ex).__name__}: {ex}")
                continue
            if not df.equals(original) or list(df.columns) != list(original.columns):
                r.mal("`agregar_variables` modificó el DataFrame que recibió: trabaja sobre una copia (`df.copy()`).")
                continue
            if not isinstance(res, pd.DataFrame) or len(res) != len(df):
                r.mal(f"`agregar_variables` {texto} debería devolver un DataFrame con {len(df)} filas.")
                continue
            faltan = [c for c in ["ratio_deuda", "log_ingreso", "tramo_edad"] + list(df.columns) if c not in res.columns]
            if faltan:
                r.mal(f"A lo que devuelve `agregar_variables` le faltan las columnas {faltan}.")
                continue
            ratio, logi, tramo = _variables_ref(df)
            if not _cerca_lista(res["ratio_deuda"].tolist(), ratio, 1e-9):
                r.mal(f"`ratio_deuda` no es correcta {texto}: es la deuda total dividida por el ingreso mensual.")
            elif not _cerca_lista(res["log_ingreso"].tolist(), logi, 1e-9):
                r.mal(f"`log_ingreso` no es correcta {texto}: usa `np.log1p` del ingreso mensual.")
            elif [str(t) for t in res["tramo_edad"]] != tramo:
                r.mal(f"`tramo_edad` no es correcta {texto}: revisa los cortes y las etiquetas (`TRAMOS`); 30 va en \"hasta 30\" y 45 en \"31-45\".")
            else:
                r.ok(f"`agregar_variables` funciona {texto}.")
    p = _particion()
    if p is not None:
        xs, xt, ys, yt = p
        for nombre, datos in (("train_fe", xs), ("test_fe", xt)):
            v = r.var(nombre)
            if v is _FALTA:
                continue
            ratio, logi, _ = _variables_ref(datos)
            if not isinstance(v, pd.DataFrame) or not v.index.equals(datos.index) or "ratio_deuda" not in v or "log_ingreso" not in v \
                    or not _cerca_lista(v["ratio_deuda"].tolist(), ratio, 1e-9) or not _cerca_lista(v["log_ingreso"].tolist(), logi, 1e-9):
                r.mal(f"`{nombre}` debería ser `agregar_variables` aplicada a su conjunto.")
            else:
                r.ok(f"`{nombre}` es correcto.")
        tf, sf = globals().get("train_fe"), globals().get("test_fe")
        _esc(r, "auc_base", _auc_logistica(xs, ys, xt, yt, NUMERICAS), "logística con `NUMERICAS` escaladas (escalador ajustado en entrenamiento), evaluada en prueba", tol=1e-6)
        if isinstance(tf, pd.DataFrame) and isinstance(sf, pd.DataFrame) and all(c in tf and c in sf for c in NUEVAS_NUM):
            _esc(r, "auc_nuevas", _auc_logistica(tf, ys, sf, yt, NUEVAS_NUM), "logística con `NUEVAS_NUM` escaladas, evaluada en prueba", tol=1e-6)
        else:
            r.mal("Para revisar `auc_nuevas` necesito `train_fe` y `test_fe` con las columnas de `NUEVAS_NUM`.")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_ratio": "8baa8f536c27bed66fdc9c147dba5054cd6d56bd432017f36d110fae0ec28b4e",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    p = _particion()
    if p is None:
        r.mal("Primero resuelve el ejercicio 1.")
        r.fin()
        return
    xs, xt, ys, yt = p
    cols = NUMERICAS + ["dias_mora_actual"]
    aucs = {c: _auc(ys.tolist(), xs[c].tolist()) for c in cols}
    orden = sorted(aucs, key=lambda k: -aucs[k])
    _ser(r, "auc_por_variable", [aucs[k] for k in orden], "el AUC de cada columna de `NUMERICAS` y de `dias_mora_actual` usada tal cual como puntaje, en entrenamiento, de mayor a menor", indice=orden, tol=1e-9)
    fuga = r.var("columna_fuga", str)
    if fuga is not _FALTA:
        if fuga.strip() == orden[0]:
            r.ok("`columna_fuga` es correcta.")
        elif fuga.strip() in cols:
            r.mal(f"`{fuga.strip()}` no es la columna con un AUC sospechosamente alto; mira el primer lugar de `auc_por_variable`.")
        else:
            r.mal("`columna_fuga` debería ser el nombre de una de las columnas numéricas.")
    _esc(r, "auc_con_fuga", _auc_logistica(xs, ys, xt, yt, NUMERICAS + ["dias_mora_actual"]),
         "logística con `NUMERICAS` más la columna con fuga, escaladas, evaluada en prueba", tol=1e-6)
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_disponible": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def _revisar_pipe(r, nombre, tipo_modelo, tf, ys):
    pipe = r.var(nombre)
    if pipe is _FALTA:
        return None
    if type(pipe).__name__ != "Pipeline" or list(pipe.named_steps) != ["prep", "modelo"]:
        r.mal(f"`{nombre}` debería ser un `Pipeline` con dos pasos llamados \"prep\" y \"modelo\".")
        return None
    prep, m = pipe.named_steps["prep"], pipe.named_steps["modelo"]
    if type(prep).__name__ != "ColumnTransformer" or type(m).__name__ != tipo_modelo:
        r.mal(f"En `{nombre}`, \"prep\" debería ser un `ColumnTransformer` y \"modelo\" un `{tipo_modelo}`.")
        return None
    if _usa_fuga(pipe):
        r.mal(f"`{nombre}` usa una columna con fuga o el objetivo: quítala del `ColumnTransformer`.")
        return None
    trans = {n: (type(t).__name__, list(c) if isinstance(c, list) else c) for n, t, c in prep.transformers}
    if trans != {"num": ("StandardScaler", NUEVAS_NUM), "cat": ("OneHotEncoder", CATEGORICAS)}:
        r.mal(f"El \"prep\" de `{nombre}` debería tener \"num\" (`StandardScaler` sobre `NUEVAS_NUM`) y \"cat\" (`OneHotEncoder` sobre `CATEGORICAS`).")
        return None
    if prep.transformers[1][1].handle_unknown != "ignore":
        r.mal(f"El `OneHotEncoder` de `{nombre}` debería usar `handle_unknown=\"ignore\"`.")
        return None
    if not hasattr(m, "classes_"):
        r.mal(f"`{nombre}` todavía no está entrenado: llama a `{nombre}.fit(train_fe, y_train)`.")
        return None
    r.ok(f"`{nombre}` tiene los pasos correctos y está entrenado.")
    return pipe


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    p = _particion()
    tf, sf = globals().get("train_fe"), globals().get("test_fe")
    if p is None or not isinstance(tf, pd.DataFrame) or not isinstance(sf, pd.DataFrame):
        r.mal("Primero resuelve los ejercicios 1 y 3 (`train_fe` y `test_fe`).")
        r.fin()
        return
    xs, xt, ys, yt = p
    pipe = _revisar_pipe(r, "pipe", "LogisticRegression", tf, ys)
    if pipe is not None:
        _esc(r, "auc_cv_pipe", _cv_manual(pipe, tf, ys), "el promedio de `cross_val_score` con el `StratifiedKFold` pedido sobre `train_fe`", tol=1e-6)
        _esc(r, "auc_test_pipe", _auc(yt.tolist(), pipe.predict_proba(sf)[:, 1]), "el AUC en prueba de `pipe` (usa `test_fe`)", tol=1e-9)
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_reajusta": "6c5fb3b25e6ba7dcf12155440e0c51b36a0492e24123968a10f8c31c304ebb6e",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    p = _particion()
    tf, sf = globals().get("train_fe"), globals().get("test_fe")
    if p is None or not isinstance(tf, pd.DataFrame) or not isinstance(sf, pd.DataFrame) or not hasattr(globals().get("pipe"), "predict_proba"):
        r.mal("Primero resuelve los ejercicios 1, 3 y 5.")
        r.fin()
        return
    xs, xt, ys, yt = p
    pl = _revisar_pipe(r, "pipe_lgbm", "LGBMClassifier", tf, ys)
    if pl is None:
        r.fin()
        return
    m = pl.named_steps["modelo"]
    if (m.n_estimators, m.learning_rate, m.num_leaves, m.random_state) != (200, 0.05, 15, 42):
        r.mal("El LightGBM de `pipe_lgbm` debería tener `n_estimators=200`, `learning_rate=0.05`, `num_leaves=15` y `random_state=42`.")
        r.fin()
        return
    cv_l = _cv_manual(pl, tf, ys)
    _esc(r, "auc_cv_lgbm", cv_l, "el promedio de `cross_val_score` de `pipe_lgbm` con el mismo `StratifiedKFold`", tol=1e-6)
    cv_p = _cv_manual(pipe, tf, ys)
    mejor = r.var("mejor_pipe")
    if mejor is not _FALTA:
        if mejor is not (pl if cv_l > cv_p else pipe):
            r.mal("`mejor_pipe` debería ser el pipeline con mayor AUC de **validación cruzada** (`pipe` o `pipe_lgbm`).")
        else:
            r.ok("`mejor_pipe` es el que ganó en validación cruzada.")
    pn = r.var("prob_nuevas")
    if pn is not _FALTA and mejor is not _FALTA:
        pn = np.asarray(pn, dtype=float)
        if pn.shape != (len(solicitudes_nuevas),):
            r.mal(f"`prob_nuevas` tiene forma {pn.shape} y se esperaba ({len(solicitudes_nuevas)},): una probabilidad por solicitud.")
        elif np.isnan(pn).any() or pn.min() < 0 or pn.max() > 1:
            r.mal("`prob_nuevas` debería tener probabilidades entre 0 y 1, sin vacíos.")
        elif hasattr(mejor, "predict_proba") and not np.allclose(pn, mejor.predict_proba(agregar_variables(solicitudes_nuevas))[:, 1], atol=1e-9):
            r.mal("`prob_nuevas` no coincide con las probabilidades de incumplir que da `mejor_pipe` para `agregar_variables(solicitudes_nuevas)`.")
        else:
            r.ok("`prob_nuevas` es correcto, incluso con categorías que el modelo nunca vio.")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    b = r.var("busqueda_pipe")
    tf, p = globals().get("train_fe"), _particion()
    if b is not _FALTA:
        if type(b).__name__ != "GridSearchCV" or not hasattr(b, "best_params_") or type(b.estimator).__name__ != "Pipeline":
            r.mal("`busqueda_pipe` debería ser un `GridSearchCV` entrenado sobre `pipe`.")
        elif b.param_grid != {"modelo__C": [0.01, 0.1, 1, 10]} or b.scoring != "roc_auc" or getattr(b.cv, "n_splits", None) != 5:
            r.mal("Revisa la grilla (`modelo__C`), `scoring=\"roc_auc\"` y los 5 pliegues de `busqueda_pipe`.")
        elif p is None or not isinstance(tf, pd.DataFrame) or len(b.cv_results_["params"]) != 4:
            r.mal("`busqueda_pipe` debería entrenarse con `train_fe` y `y_train`.")
        else:
            res = b.cv_results_
            r.ok("`busqueda_pipe` probó los cuatro valores de `C`.")
            _esc(r, "mejor_C", res["params"][int(np.argmax(res["mean_test_score"]))]["modelo__C"], "el valor de `C` con mayor AUC de validación cruzada", tol=1e-12)
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
`prestamos`: 3000 préstamos otorgados, con el tipo de empleo, la región y el producto del cliente, su ingreso mensual y su deuda total (en soles), los meses en su empleo, su edad, los **días de mora que registra hoy** el préstamo y si terminó **incumpliendo** (1) o no (0).

`solicitudes_nuevas`: 4 personas que están pidiendo un préstamo ahora. Fíjate que traen algunas categorías que no aparecen en `prestamos`.

Listas de columnas: `CATEGORICAS`, `NUMERICAS`, `NUEVAS_NUM` (las numéricas que crearás en el ejercicio 3) y `TRAMOS` (etiquetas de tramos de edad).

In [ ]:
print(prestamos.head(), "\n")
print(prestamos["incumple"].mean().round(3), "\n")
for col in CATEGORICAS:
    print(prestamos[col].value_counts().to_dict())
print("\n", solicitudes_nuevas)

---
## 1. Codificar categorías: one-hot

### 📘 Concepto
Los modelos solo entienden números. Una columna de texto como `region` se convierte con **one-hot**: una columna nueva por categoría, con 1 si la fila es de esa categoría y 0 si no. No se usa 1, 2, 3... porque eso inventaría un orden ("Sur" no es mayor que "Norte").

`OneHotEncoder` de scikit-learn **aprende las categorías** con `fit` y las aplica con `transform`:
- se ajusta **solo con el entrenamiento**, igual que un modelo;
- `handle_unknown="ignore"` hace que una categoría nunca vista quede como una fila de ceros en lugar de dar error;
- `sparse_output=False` devuelve un array normal en vez de una matriz dispersa;
- `get_feature_names_out()` da los nombres de las columnas creadas.

`pd.get_dummies` hace algo parecido, pero no recuerda las categorías: si el conjunto de prueba trae otras, las columnas no coinciden.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

colores_ej = pd.DataFrame({"color": ["rojo", "azul", "rojo", "verde"]})
cod_ej = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(colores_ej)
print(cod_ej.get_feature_names_out())
print(cod_ej.transform(colores_ej))
print(cod_ej.transform(pd.DataFrame({"color": ["azul", "morado"]})))   # "morado" no se vio: todo ceros

### ✍️ Tu turno · Ejercicio 1: one-hot
**Parte A.**
1. `X` (todas las columnas de `prestamos` menos `incumple`) e `y` (`incumple`), y `X_train`, `X_test`, `y_train`, `y_test` con 25 % para prueba, `random_state=42` y estratificado.
2. `codificador`: un `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` ajustado con `X_train[CATEGORICAS]`.
3. `nombres_onehot`: la **lista** de nombres de las columnas creadas.
4. `cat_train`: un DataFrame con la transformación de `X_train[CATEGORICAS]`, `nombres_onehot` como columnas y el índice de `X_train`.
5. `cat_nuevas`: la transformación de `solicitudes_nuevas[CATEGORICAS]`.

**Parte B.** Predice **sin ejecutar**: `pred_unos_fila0` = cuántos 1 tiene la primera fila de `cat_nuevas` (un entero). Mira qué región trae esa solicitud.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Ajusta con `codificador = OneHotEncoder(...).fit(X_train[CATEGORICAS])` y transforma con `codificador.transform(...)`.
</details>

<details><summary>💡 Pista 2</summary>

`pd.DataFrame(codificador.transform(X_train[CATEGORICAS]), columns=nombres_onehot, index=X_train.index)`. Para la lista, `list(codificador.get_feature_names_out())`.
</details>

---
## 2. Escalar variables numéricas

### 📘 Concepto
El ingreso se mide en miles de soles y la edad en decenas de años. Modelos como la regresión logística (con regularización) o KNN se ven afectados por esas escalas. `StandardScaler` resta la media y divide por la desviación estándar de cada columna: quedan centradas en 0 y con desviación 1.

La regla de oro es la misma que con el codificador: **`fit` solo con el entrenamiento** y `transform` en entrenamiento y prueba. Si calculas la media con todos los datos, la prueba "filtra" información al entrenamiento. Por eso la prueba escalada **no** queda con media exactamente 0: usa la media del entrenamiento.

Los árboles, Random Forest y LightGBM no necesitan escalado: solo comparan valores con umbrales.

In [ ]:
from sklearn.preprocessing import StandardScaler

a_ej = np.array([[10.0], [20.0], [30.0]])
b_ej = np.array([[40.0]])
esc_ej = StandardScaler().fit(a_ej)
print(esc_ej.mean_, esc_ej.scale_)
print(esc_ej.transform(a_ej).ravel(), esc_ej.transform(b_ej).ravel())

### ✍️ Tu turno · Ejercicio 2: escalar sin hacer trampa
**Parte A.**
1. `escalador`: un `StandardScaler` ajustado con `X_train[NUMERICAS]`.
2. `num_train` y `num_test`: `X_train[NUMERICAS]` y `X_test[NUMERICAS]` transformados con `escalador`.

Mira `num_train.mean(axis=0).round(3)`, `num_train.std(axis=0).round(3)` y lo mismo para `num_test`.

**Parte B.** Responde en `pred_media_test_cero` con `"sí"` o `"no"`: ¿las columnas de `num_test` quedan con media exactamente 0?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Un solo `fit`, con el entrenamiento. Después, dos `transform`.
</details>

<details><summary>💡 Pista 2</summary>

`escalador = StandardScaler().fit(X_train[NUMERICAS])` y `num_test = escalador.transform(X_test[NUMERICAS])`.
</details>

---
## 3. Crear variables nuevas

### 📘 Concepto
Muchas veces el modelo mejora más con **mejores variables** que con un algoritmo más sofisticado. Ideas frecuentes:
- **Proporciones**: una deuda de 10 000 soles no pesa igual con un ingreso de 2000 que de 20 000. La deuda dividida por el ingreso mide mejor la carga.
- **Logaritmos**: los ingresos tienen una cola larga de valores muy altos. `np.log1p(x)` (logaritmo de 1 + x, que acepta ceros) comprime esa cola.
- **Tramos**: `pd.cut(serie, bins=[...], labels=[...])` agrupa una variable numérica en intervalos. Por defecto cada intervalo incluye su borde derecho: con `bins=[0, 30, 45]`, 30 cae en el primer tramo.

Conviene escribir estas transformaciones en una **función** para aplicarlas igual al entrenamiento, a la prueba y a los datos nuevos, y trabajar sobre una copia para no modificar el original.

In [ ]:
def agregar_ej(df):
    df = df.copy()
    df["precio_por_unidad"] = df["monto"] / df["unidades"]
    df["log_monto"] = np.log1p(df["monto"])
    df["tamaño"] = pd.cut(df["unidades"], bins=[0, 2, 10, 1000], labels=["chica", "mediana", "grande"])
    return df

ventas_ej = pd.DataFrame({"monto": [0.0, 50.0, 1200.0], "unidades": [2, 5, 40]})
print(agregar_ej(ventas_ej))
print(ventas_ej.columns.tolist())   # el original no cambió

### ✍️ Tu turno · Ejercicio 3: variables que cuentan mejor la historia
**Parte A.**
1. `agregar_variables(df)`: una función que devuelve una **copia** de `df` con tres columnas nuevas:
   - `ratio_deuda`: deuda total dividida por el ingreso mensual;
   - `log_ingreso`: `np.log1p` del ingreso mensual;
   - `tramo_edad`: la edad en tramos con `bins=[0, 30, 45, 60, 120]` y `labels=TRAMOS`.
2. `train_fe` y `test_fe`: la función aplicada a `X_train` y `X_test`.
3. `auc_base`: el AUC en prueba de una regresión logística (`max_iter=1000`) con `NUMERICAS` escaladas (usa tus `num_train` y `num_test`).
4. `auc_nuevas`: lo mismo con las columnas `NUEVAS_NUM` de `train_fe` y `test_fe`, escaladas con un `StandardScaler` nuevo ajustado en `train_fe`.

¿Mejoró el modelo con las variables nuevas, usando la misma cantidad de columnas?

**Parte B.** Predice **sin ejecutar**: `pred_ratio` = el `ratio_deuda` de alguien que gana 2000 al mes y debe 5000 (un número).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Copia la estructura de `agregar_ej`. Para los AUC, entrena `LogisticRegression(max_iter=1000)` y usa `roc_auc_score` con `predict_proba(...)[:, 1]`.
</details>

<details><summary>💡 Pista 2</summary>

`esc_fe = StandardScaler().fit(train_fe[NUEVAS_NUM])`, y luego transforma `train_fe[NUEVAS_NUM]` y `test_fe[NUEVAS_NUM]` con él.
</details>

---
## 4. Data leakage: cuando el modelo hace trampa

### 📘 Concepto
Hay **fuga de información** (*data leakage*) cuando el modelo usa algo que **no estaría disponible** al momento de predecir. El resultado es un AUC espectacular en tus pruebas y un modelo inútil en la realidad. Las formas más comunes:
- **Variables del futuro**: datos que se registran después del evento que quieres predecir, o que son una consecuencia de él.
- **Preprocesar con todos los datos**: ajustar un escalador, un codificador o elegir variables usando también la prueba.
- **Filas repetidas** entre entrenamiento y prueba.

Una pista práctica: si una sola variable separa casi perfectamente las clases, sospecha. Puedes medirlo usando la columna tal cual como puntaje en `roc_auc_score`: un AUC cercano a 1 es una alarma, cercano a 0,5 es una variable sin señal, y menor que 0,5 indica una relación inversa (más valor, menos riesgo).

La pregunta clave siempre es: **¿conozco este dato en el momento en que tengo que decidir?**

In [ ]:
from sklearn.metrics import roc_auc_score

y_ej = pd.Series([0, 0, 1, 1, 0, 1])
senales_ej = pd.DataFrame({"normal": [3, 5, 4, 6, 2, 5], "sospechosa": [0, 0, 30, 45, 0, 60]})
print({col: round(roc_auc_score(y_ej, senales_ej[col]), 3) for col in senales_ej})

### ✍️ Tu turno · Ejercicio 4: encuentra la fuga
**Parte A.**
1. `auc_por_variable`: una Series con el AUC de usar cada columna de `NUMERICAS` y `"dias_mora_actual"` directamente como puntaje, en el **entrenamiento**, ordenada de mayor a menor.
2. `columna_fuga`: el nombre (texto) de la columna sospechosa.
3. `auc_con_fuga`: el AUC en prueba de una regresión logística (`max_iter=1000`) con `NUMERICAS` más esa columna, escaladas con un `StandardScaler` ajustado en el entrenamiento. Compáralo con `auc_base`.

Escribe en una celda de texto por qué ese modelo no serviría para decidir a quién prestar.

**Parte B.** Responde en `pred_disponible` con `"sí"` o `"no"`: cuando una persona pide un préstamo, ¿ya se conoce el valor de `columna_fuga` para **ese** préstamo?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Recorre la lista `NUMERICAS + ["dias_mora_actual"]` y calcula `roc_auc_score(y_train, X_train[col])` para cada una.
</details>

<details><summary>💡 Pista 2</summary>

Arma un diccionario `{columna: auc}`, conviértelo en Series y ordénalo con `sort_values(ascending=False)`.
</details>

---
## 5. `Pipeline` y `ColumnTransformer`

### 📘 Concepto
Hacer cada paso a mano es fácil de equivocar: olvidar escalar la prueba, ajustar el codificador con todo... Un **`ColumnTransformer`** aplica un preprocesamiento distinto a cada grupo de columnas, y un **`Pipeline`** encadena ese preprocesamiento con el modelo en un solo objeto:

```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

prep = ColumnTransformer([
    ("num", StandardScaler(), columnas_numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
])
pipe = Pipeline([("prep", prep), ("modelo", LogisticRegression(max_iter=1000))])
pipe.fit(X_train, y_train)          # ajusta prep y modelo solo con el entrenamiento
pipe.predict_proba(X_test)          # transforma y predice en un paso
```

Las columnas que no nombras en el `ColumnTransformer` se descartan: así dejas fuera la fuga. La gran ventaja aparece con la **validación cruzada**: `cross_val_score(pipe, ...)` reajusta el preprocesamiento dentro de cada pliegue, así que ningún pliegue de validación contamina el escalado.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

tiendas_ej = pd.DataFrame({"zona": ["A", "B", "A", "C", "B", "A"], "ventas": [10.0, 50.0, 12.0, 40.0, 55.0, 9.0], "nota": ["x"] * 6})
exito_ej = pd.Series([0, 1, 0, 1, 1, 0])
pipe_ej = Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), ["ventas"]), ("cat", OneHotEncoder(handle_unknown="ignore"), ["zona"])])),
    ("modelo", LogisticRegression(max_iter=1000)),
]).fit(tiendas_ej, exito_ej)
print(pipe_ej.named_steps["prep"].get_feature_names_out())    # "nota" quedó fuera
print(pipe_ej.predict_proba(pd.DataFrame({"zona": ["Z"], "ventas": [45.0], "nota": ["x"]}))[:, 1].round(3))

### ✍️ Tu turno · Ejercicio 5: todo en un pipeline
**Parte A.**
1. `preprocesador`: un `ColumnTransformer` con `("num", StandardScaler(), NUEVAS_NUM)` y `("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)`.
2. `pipe`: un `Pipeline` con los pasos `("prep", preprocesador)` y `("modelo", LogisticRegression(max_iter=1000))`.
3. `auc_cv_pipe`: el promedio de `cross_val_score(pipe, train_fe, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc")`.
4. Entrena `pipe` con `train_fe` y `y_train`; `auc_test_pipe`: su AUC en `test_fe`.

¿Cuánto aportaron las categorías respecto de `auc_nuevas`?

**Parte B.** Responde en `pred_reajusta` con `"sí"` o `"no"`: dentro de `cross_val_score`, ¿el `StandardScaler` se vuelve a ajustar en cada pliegue solo con los datos de entrenamiento de ese pliegue?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Copia la estructura del concepto con `NUEVAS_NUM` y `CATEGORICAS`. `cross_val_score` devuelve un array: saca su `.mean()`.
</details>

<details><summary>💡 Pista 2</summary>

`cross_val_score` entrena copias de `pipe`, así que igual tienes que llamar a `pipe.fit(train_fe, y_train)` antes de `pipe.predict_proba(test_fe)`.
</details>

---
## 🏋️ Reto final: elegir modelo y puntuar solicitudes nuevas
1. `pipe_lgbm`: el mismo preprocesamiento (un `ColumnTransformer` nuevo, igual al de `pipe`) con `("modelo", LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, random_state=42, verbose=-1))`, entrenado con `train_fe`.
2. `auc_cv_lgbm`: su AUC promedio de validación cruzada, con el mismo `StratifiedKFold` del ejercicio 5.
3. `mejor_pipe`: `pipe` o `pipe_lgbm`, el que tenga **mayor AUC de validación cruzada**.
4. `prob_nuevas`: la probabilidad de incumplir de cada persona de `solicitudes_nuevas` según `mejor_pipe` (recuerda aplicar `agregar_variables` antes).

¿Ganó el modelo más complejo? ¿Qué pasó con las categorías que el modelo nunca vio?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Para no compartir el mismo `ColumnTransformer` entre dos pipelines, crea uno nuevo con la misma definición.
</details>

<details><summary>💡 Pista 2</summary>

`mejor_pipe = pipe_lgbm if auc_cv_lgbm > auc_cv_pipe else pipe` y `mejor_pipe.predict_proba(agregar_variables(solicitudes_nuevas))[:, 1]`.
</details>

---
## 🚀 Nivel pro (opcional): ajustar hiperparámetros dentro del pipeline
`GridSearchCV` acepta un pipeline completo. Para nombrar un hiperparámetro de un paso se usa `paso__parámetro`, con dos guiones bajos: por ejemplo, `modelo__C` es el `C` de la regresión logística (menor `C` = más regularización).

Crea `busqueda_pipe`: un `GridSearchCV` sobre `pipe` con la grilla `{"modelo__C": [0.01, 0.1, 1, 10]}`, el `StratifiedKFold(5, shuffle=True, random_state=42)` y `scoring="roc_auc"`, entrenado con `train_fe` y `y_train`. Guarda en `mejor_C` el valor elegido.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Codificar categorías con `OneHotEncoder` y explicar `handle_unknown="ignore"`.
- [ ] Escalar con `StandardScaler` ajustando solo con el entrenamiento.
- [ ] Crear proporciones, logaritmos y tramos en una función que no modifica el original.
- [ ] Reconocer una fuga de información y explicar por qué infla las métricas.
- [ ] Armar un `Pipeline` con `ColumnTransformer` y validarlo con `cross_val_score`.
- [ ] Usar un pipeline entrenado para puntuar datos nuevos.

**Próxima sesión (S22):** flujo de competencia: validación honesta, leaderboard público y privado, archivo de envío y registro de experimentos.